### Image Clustering - Improving Validation Set

A naive training/validation split will contain many near-duplicate images, as well as images that are not near duplicates but are very similar and are taken from the same drone on the same day in the same location. This leads to severely inflated validation set performance and a large degradation on the test set. To improve the validation set, I decided to use a deep learning based approach to group the images in the training set into clusters based on visual similarity. My first approach used an ImageNet-trained ResNet50 model backbone to extract features. I then used HBDSCAN to group the images into clusters. This approach did separate images into visually distinct, but it didn't do a good enough job given the size and complexity of the dataset. I tried manually fixing many of the outliers and misclassified images, but it proved infeasible. So, I did some research to try to find what a better approach would be. What the internet seems to recommend is to use the self-supervised transformer-based DINOv2 model to extract features, as this should give far improved results especially for large aerial drone images. Below, I implement an image clustering algorithm using DINOv2 and visualize the results.

In [119]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hdbscan import HDBSCAN
from torch.utils.data import Dataset as TorchDataset, DataLoader
import albumentations as A
import cv2
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import math
import time

In [120]:
@dataclass
class Config:
    images_root_folder: str
    masks_root_folder: str
    cluster_labels_filepath: str
    class_values_filepath: str
    device: str

    # noinspection PyAttributeOutsideInit
    def init(self):
        self.image_height = 224
        self.image_width = 392
        self.image_dims = (self.image_height, self.image_width)
        self.image_transforms = A.Compose([
            A.Resize(height=self.image_height, width=self.image_width, interpolation=cv2.INTER_AREA),
            A.Normalize(imagenet_mean_tuple, imagenet_std_tuple),
            A.ToTensorV2(),
        ])
        self.model_repo = 'facebookresearch/dinov2'
        self.model_name = 'dinov2_vitb14'
        self.min_cluster_size = 10
        self.min_samples = 5
        self.cluster_selection_epsilon = 150.0
        self.cluster_selection_method = 'eom'
        self.pca_components = None
        self.separate_outliers = True
        self.num_workers = 4 if self.device == 'cuda' else 0
        self.pin_memory = self.num_workers > 0
        self.batch_size = 16
        self.image_ids = [file.stem for file in Path(self.images_root_folder).glob('*.jpg')]
        self.embeddings_file = f'{self.images_root_folder}embeddings.npy'
        self.cluster_labels_file = f'{self.images_root_folder}cluster_labels.csv'
        self.cluster_visualization_file = f'{self.images_root_folder}cluster_visualization.png'

config: Config = None

In [121]:
local_config = Config(
    images_root_folder='data/images/',
    masks_root_folder='data/masks/',
    cluster_labels_filepath='data/cluster_labels.csv',
    class_values_filepath='data/ids_with_class_values.csv',
    device='cpu',
)

In [122]:
imagenet_mean_tuple = (0.485, 0.456, 0.406)
imagenet_std_tuple = (0.229, 0.224, 0.225)

In [123]:
class ClusterDataset(TorchDataset):
    def __len__(self):
        return len(config.image_ids)

    def __getitem__(self, idx):
        image_id = config.image_ids[idx]
        image = cv2.cvtColor(cv2.imread(f'{config.images_root_folder}{image_id}.jpg'), cv2.COLOR_BGR2RGB)
        return config.image_transforms(image=image)['image']

In [124]:
def visualize_clusters(
    image_ids,
    cluster_labels,
    max_clusters = 70,
    samples_per_cluster = 10,
    subfolders_format = False,
):
    unique_clusters = np.unique(cluster_labels)
    n_clusters = len(unique_clusters)

    # Sort clusters by size (largest first)
    cluster_sizes = [(c, np.sum(cluster_labels == c)) for c in unique_clusters]
    cluster_sizes.sort(key=lambda x: -x[1])

    clusters_to_show = [c for c, _ in cluster_sizes[:max_clusters]]
    n_rows = min(max_clusters, n_clusters)

    fig, axes = plt.subplots(n_rows, samples_per_cluster, figsize=(samples_per_cluster * 2.5, n_rows * 2.5))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    image_ids = np.array(image_ids)

    for row_idx, cluster_id in enumerate(clusters_to_show):
        cluster_mask = cluster_labels == cluster_id
        cluster_size = np.sum(cluster_mask)
        cluster_image_ids = image_ids[cluster_mask]

        if cluster_size > 0:
            sample_indices = np.linspace(0, cluster_size - 1, min(samples_per_cluster, cluster_size), dtype=int)
        else:
            sample_indices = []

        for col_idx in range(samples_per_cluster):
            ax = axes[row_idx, col_idx]

            if col_idx < len(sample_indices):
                image_id = cluster_image_ids[sample_indices[col_idx]]
                image_path = f'{config.images_root_folder}{cluster_id}/{image_id}.jpg' if subfolders_format else f'{config.images_root_folder}{image_id}.jpg'
                img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
                ax.imshow(img)

            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

            if col_idx == 0:
                ax.set_ylabel(f"Cluster {cluster_id}\n(n={cluster_size})", fontsize=8)

    plt.suptitle(f"Sample Images from {n_rows} Clusters (of {n_clusters} total)", fontsize=12)
    plt.tight_layout()
    plt.savefig(config.cluster_visualization_file, dpi=150, bbox_inches="tight")
    print(f"Visualization saved to {config.cluster_visualization_file}")
    plt.show()

In [125]:
def generate_image_clusters():
    start_time = time.time()

    # Check for cached embeddings
    embeddings_path = Path(config.embeddings_file)
    if embeddings_path.exists():
        print(f'Loading cached embeddings from {embeddings_path}')
        embeddings = np.load(embeddings_path)
        if len(embeddings) != len(config.image_ids):
            print(f'Cached embeddings count ({len(embeddings)}) != image count '
                  f'({len(config.image_ids)}). Re-extracting...')
            embeddings = None
        else:
            print(f'Loaded {len(embeddings)} embeddings from cache')
    else:
        embeddings = None

    if embeddings is None:
        print('Starting data prep and model loading')
        dataset = ClusterDataset()
        loader = DataLoader(dataset, shuffle=False, batch_size=config.batch_size, num_workers=config.num_workers, pin_memory=config.pin_memory)

        model = torch.hub.load(config.model_repo, config.model_name).to(config.device)
        model.eval()

        batch_embeddings_list = []
        num_batches = math.ceil(len(loader.dataset) / config.batch_size)
        with torch.no_grad():
            for batch_number, batch in enumerate(loader):
                print(f't={time.time() - start_time:.1f}s: Extracting batch {batch_number + 1}/{num_batches}')
                batch = batch.to(config.device)
                batch_embeddings = model(batch)
                batch_embeddings_list.append(batch_embeddings.cpu().numpy())

        embeddings = np.vstack(batch_embeddings_list)
        np.save(config.embeddings_file, embeddings)
        print(f'Embeddings saved to {config.embeddings_file}')

    embeddings_dim = embeddings.shape[1]

    scaler = StandardScaler()
    embeddings = scaler.fit_transform(embeddings)

    if config.pca_components is not None:
        effective_pca = min(config.pca_components, embeddings_dim)
        pca = PCA(n_components=effective_pca)
        embeddings = pca.fit_transform(embeddings)
        explained_var = pca.explained_variance_ratio_.sum()
        print(f'PCA: {effective_pca} components explain {explained_var:.1%} of variance')

    clusterer = HDBSCAN(
        min_cluster_size=config.min_cluster_size,
        min_samples=config.min_samples,
        cluster_selection_epsilon=config.cluster_selection_epsilon,
        cluster_selection_method=config.cluster_selection_method,
        metric='euclidean',
    )
    clusterer.fit(embeddings)
    clusterer.condensed_tree_.plot()
    cluster_labels = clusterer.fit_predict(embeddings)
    unique, counts = np.unique(cluster_labels, return_counts=True)
    sizes = sorted(counts, reverse=True)

    print(f'Cluster sizes (sorted): {sizes[:20]}{'...' if len(sizes) > 20 else ''}')
    print(f'Min: {min(sizes)}, Max: {max(sizes)}, Mean: {np.mean(sizes):.1f}')

    df = pd.DataFrame({'id': config.image_ids, 'cluster': cluster_labels})
    df.to_csv(config.cluster_labels_file, index=False)
    print(f'Cluster labels saved to {config.cluster_labels_file}')

    visualize_clusters(
        image_ids = config.image_ids,
        cluster_labels = cluster_labels,
    )

In [126]:
def group_images_into_cluster_folders():
    cluster_labels_df = pd.read_csv(config.cluster_labels_file)
    unique_clusters = cluster_labels_df['cluster'].unique()
    for cluster_id in unique_clusters:
        Path(f'{config.images_root_folder}cluster_{cluster_id}').mkdir(exist_ok=True)
    moved = 0
    for image_id, cluster_id in cluster_labels_df.itertuples(index=False):
        shutil.move(f'{config.images_root_folder}{image_id}.jpg', f'{config.images_root_folder}cluster_{cluster_id}/{image_id}.jpg')
        moved += 1
    print(f'Moved {moved} images into {len(unique_clusters)} folders.')

In [127]:
def ungroup_images_in_folder(images_folder: str):
    images_folder = Path(images_folder)
    subfolders = [item for item in images_folder.iterdir() if item.is_dir()]
    moved = 0
    deleted = 0
    for subfolder in subfolders:
        for file in subfolder.iterdir():
            shutil.move(str(file), str(images_folder / file.name))
            moved += 1
        subfolder.rmdir()
        deleted += 1
    print(f"Moved {moved} files back to the main folder. Deleted {deleted} subfolders.")

In [128]:
def split_images_folder_into_train_and_test():
    images_folder = Path(config.images_root_folder)
    train_folder = images_folder / 'train'
    test_folder = images_folder / 'test'
    train_folder.mkdir(exist_ok=True)
    test_folder.mkdir(exist_ok=True)

    string_ids = [str(image_id) for image_id in config.image_ids]
    train_count = 0
    test_count = 0
    for file in images_folder.glob('*.jpg'):
        image_id = file.stem
        if image_id in string_ids:
            shutil.move(str(file), str(train_folder / file.name))
            train_count += 1
        else:
            shutil.move(str(file), str(test_folder / file.name))
            test_count += 1

    print(f'Moved {train_count} images to {train_folder}')
    print(f'Moved {test_count} images to {test_folder}')

In [129]:
def generate_cluster_labels_from_subfolders():
    images_root = Path(config.images_root_folder)
    rows = []
    for subfolder in sorted(images_root.iterdir()):
        if not subfolder.is_dir():
            continue
        for file in subfolder.glob('*.jpg'):
            rows.append((file.stem, subfolder.name))
    df = pd.DataFrame(rows, columns=['id', 'cluster'])
    df.to_csv(config.cluster_labels_file, index=False)
    n_clusters = df['cluster'].nunique()
    print(f'Saved {len(df)} image labels across {n_clusters} clusters to {config.cluster_labels_file}')

    visualize_clusters(df['id'].to_numpy(), df['cluster'].to_numpy(), subfolders_format=True)

In [130]:
def analyze_clusters():
    RARE_CLASS_INDICES = range(1, 8)

    # Load data
    cluster_df = pd.read_csv(config.cluster_labels_filepath)
    cluster_df = cluster_df.rename(columns={'id': 'ImageID'})
    class_df = pd.read_csv(config.class_values_filepath)
    cluster_df['ImageID'] = cluster_df['ImageID'].astype(str)
    class_df['ImageID'] = class_df['ImageID'].astype(str)
    df = cluster_df.merge(class_df, on='ImageID', how='inner')
    assert len(df) == len(cluster_df), f"Merge lost rows: {len(cluster_df)} -> {len(df)}"

    clusters = sorted(df['cluster'].unique())
    total_images = len(df)
    print(f"Dataset: {total_images} images, {len(clusters)} clusters, 12 classes")
    print(f"Rare classes (indices {list(RARE_CLASS_INDICES)}): "
          f"{', '.join(CLASS_NAMES[i] for i in RARE_CLASS_INDICES)}")

    # Section 1: Per-cluster summary
    print("\n" + "=" * 80)
    print("SECTION 1: Per-Cluster Summary")
    print("=" * 80)
    for cluster_id in clusters:
        cdf = df[df['cluster'] == cluster_id]
        n_images = len(cdf)
        print(f"\n--- Cluster {cluster_id} ({n_images} images) ---")
        presence = (cdf[CLASS_COLS] > 0).sum()
        pixel_totals = cdf[CLASS_COLS].sum()
        cluster_total_pixels = pixel_totals.sum()
        pixel_pct = pixel_totals / cluster_total_pixels * 100 if cluster_total_pixels > 0 else pixel_totals * 0
        print(f"  {'Class':<15} {'Images':>8} {'Pixel %':>10}")
        print(f"  {'-'*35}")
        missing_classes = []
        for i in range(12):
            col = CLASS_COLS[i]
            tag = " [RARE]" if i in RARE_CLASS_INDICES else ""
            if presence[col] > 0:
                print(f"  {CLASS_NAMES[i]:<15} {presence[col]:>8} {pixel_pct[col]:>9.2f}%{tag}")
            else:
                missing_classes.append(CLASS_NAMES[i])
        if missing_classes:
            print(f"  Missing: {', '.join(missing_classes)}")

    # Section 2: Class coverage across clusters
    print("\n" + "=" * 80)
    print("SECTION 2: Class Coverage Across Clusters")
    print("=" * 80)
    for i in range(12):
        col = CLASS_COLS[i]
        tag = " [RARE]" if i in RARE_CLASS_INDICES else ""
        total_images_with_class = (df[col] > 0).sum()
        total_pixels = df[col].sum()
        containing_clusters = []
        missing_clusters = []
        for cluster_id in clusters:
            cdf = df[df['cluster'] == cluster_id]
            cluster_presence = (cdf[col] > 0).sum()
            if cluster_presence > 0:
                cluster_pixels = cdf[col].sum()
                containing_clusters.append((cluster_id, int(cluster_presence), int(cluster_pixels)))
            else:
                missing_clusters.append(cluster_id)
        print(f"\n--- {CLASS_NAMES[i]} (class_{i}){tag} ---")
        print(f"  Total: {total_images_with_class} images, {total_pixels:,} pixels")
        print(f"  Present in {len(containing_clusters)}/{len(clusters)} clusters")
        print(f"  {'Cluster':>10} {'Images':>8} {'Pixels':>14} {'% of class':>12}")
        print(f"  {'-'*46}")
        for cluster_id, img_count, px_count in sorted(containing_clusters, key=lambda x: -x[2]):
            pct = px_count / total_pixels * 100 if total_pixels > 0 else 0
            print(f"  {cluster_id:>10} {img_count:>8} {px_count:>14,} {pct:>11.1f}%")
        if missing_clusters:
            print(f"  Missing from clusters: {missing_clusters}")

    # Section 3: Feasibility assessment
    print("\n" + "=" * 80)
    print("SECTION 3: Feasibility Assessment for Cluster-Level Split")
    print("=" * 80)

    # 3a: Exclusive clusters
    print("\n--- 3a: Exclusive Clusters (sole source of a rare class) ---")
    found_exclusive = False
    for i in RARE_CLASS_INDICES:
        col = CLASS_COLS[i]
        clusters_with_class = [c for c in clusters if (df[df['cluster'] == c][col] > 0).any()]
        if len(clusters_with_class) == 1:
            found_exclusive = True
            n_imgs = (df[df['cluster'] == clusters_with_class[0]][col] > 0).sum()
            print(f"  {CLASS_NAMES[i]:<15} ONLY in cluster {clusters_with_class[0]} ({n_imgs} images)")
    if not found_exclusive:
        print("  None found — no rare class is confined to a single cluster.")

    # 3b: Concentration analysis
    print("\n--- 3b: Rare Class Concentration Analysis ---")
    print(f"  {'Class':<15} {'Clusters':>10} {'Top Cluster %':>15} {'Risk':>8}")
    print(f"  {'-'*50}")
    for i in RARE_CLASS_INDICES:
        col = CLASS_COLS[i]
        total_pixels = df[col].sum()
        if total_pixels == 0:
            print(f"  {CLASS_NAMES[i]:<15} {'N/A':>10} {'N/A':>15} {'N/A':>8}")
            continue
        cluster_pixels = [(c, df[df['cluster'] == c][col].sum()) for c in clusters if df[df['cluster'] == c][col].sum() > 0]
        n_clusters = len(cluster_pixels)
        top_pct = max(px for _, px in cluster_pixels) / total_pixels * 100
        if n_clusters <= 2 or top_pct > 80:
            risk = "HIGH"
        elif n_clusters <= 5 or top_pct > 50:
            risk = "MEDIUM"
        else:
            risk = "LOW"
        print(f"  {CLASS_NAMES[i]:<15} {n_clusters:>10} {top_pct:>14.1f}% {risk:>8}")

    # 3c: Simulated greedy split
    print("\n--- 3c: Simulated 80/20 Cluster-Level Split (greedy bin-packing) ---")
    target_train = int(total_images * 0.80)
    cluster_sizes = [(c, len(df[df['cluster'] == c])) for c in clusters]
    cluster_sizes.sort(key=lambda x: -x[1])
    train_clusters, val_clusters = [], []
    train_count = 0
    for cluster_id, size in cluster_sizes:
        if train_count + size <= target_train:
            train_clusters.append(cluster_id)
            train_count += size
        else:
            gap_if_add = abs((train_count + size) - target_train)
            gap_if_skip = abs(train_count - target_train)
            if gap_if_add < gap_if_skip:
                train_clusters.append(cluster_id)
                train_count += size
            else:
                val_clusters.append(cluster_id)
    train_mask = df['cluster'].isin(train_clusters)
    val_mask = df['cluster'].isin(val_clusters)
    train_df = df[train_mask]
    val_df = df[val_mask]
    print(f"\n  Train: {len(train_clusters)} clusters, {len(train_df)} images ({len(train_df)/total_images:.1%})")
    print(f"  Val:   {len(val_clusters)} clusters, {len(val_df)} images ({len(val_df)/total_images:.1%})")
    print(f"\n  {'Class':<15} {'Train Imgs':>12} {'Val Imgs':>12} {'Train Px%':>12} {'Val Px%':>12} {'Status':>10}")
    print(f"  {'-'*70}")
    missing_from_split = []
    for i in range(12):
        col = CLASS_COLS[i]
        tag = " [RARE]" if i in RARE_CLASS_INDICES else ""
        train_img_count = (train_df[col] > 0).sum()
        val_img_count = (val_df[col] > 0).sum()
        total_class_pixels = df[col].sum()
        if total_class_pixels > 0:
            train_px_pct = train_df[col].sum() / total_class_pixels * 100
            val_px_pct = val_df[col].sum() / total_class_pixels * 100
        else:
            train_px_pct = val_px_pct = 0
        if train_img_count == 0:
            status = "MISSING-T"
            missing_from_split.append((CLASS_NAMES[i], 'train'))
        elif val_img_count == 0:
            status = "MISSING-V"
            missing_from_split.append((CLASS_NAMES[i], 'val'))
        else:
            status = "OK"
        print(f"  {CLASS_NAMES[i]:<15} {train_img_count:>12} {val_img_count:>12} {train_px_pct:>11.1f}% {val_px_pct:>11.1f}% {status:>10}{tag}")
    print("\n  --- Conclusion ---")
    if missing_from_split:
        print("  INFEASIBLE: The following classes are missing from a split:")
        for cls_name, split in missing_from_split:
            print(f"    - {cls_name} missing from {split}")
        print("  A pure cluster-level split cannot guarantee all classes in both splits.")
        print("  Consider: rare-class-aware cluster pre-assignment to val.")
    else:
        print("  FEASIBLE: All 12 classes are represented in both train and val splits.")
        print("  A cluster-level group-aware split is viable for this dataset.")

In [131]:
CLASS_NAMES = [
    'background', 'person', 'bike', 'car', 'drone',
    'boat', 'animal', 'obstacle', 'construction',
    'vegetation', 'road', 'sky',
]
CLASS_COLS = [f'class_{i}' for i in range(12)]


def generate_train_val_splits(n_splits=5, val_ratio=0.20, random_seed=42, min_class_ratio=0.05):
    cluster_df = pd.read_csv(config.cluster_labels_filepath)
    cluster_df = cluster_df.rename(columns={'id': 'ImageID'})
    class_df = pd.read_csv(config.class_values_filepath)

    # Ensure ImageID columns are the same type (string)
    cluster_df['ImageID'] = cluster_df['ImageID'].astype(str)
    class_df['ImageID'] = class_df['ImageID'].astype(str)

    merged = cluster_df.merge(class_df, on='ImageID', how='inner')
    total_images = len(merged)
    clusters = merged['cluster'].unique()
    print(f'Total images: {total_images}, Total clusters: {len(clusters)}')

    # Precompute cluster sizes and per-cluster class image counts
    cluster_size_map = merged.groupby('cluster').size().to_dict()
    cluster_class_counts = {}
    for c in clusters:
        cdf = merged[merged['cluster'] == c]
        cluster_class_counts[c] = [(cdf[CLASS_COLS[j]] > 0).sum() for j in range(12)]

    # Precompute total images per class and proportional thresholds
    total_class_images = [(merged[CLASS_COLS[j]] > 0).sum() for j in range(12)]
    min_val_images = [max(3, round(total_class_images[j] * min_class_ratio)) for j in range(12)]

    print(f'Per-class val thresholds (min_class_ratio={min_class_ratio}):')
    for j in range(12):
        print(f'  {CLASS_NAMES[j]:<15} {total_class_images[j]:>6} total -> {min_val_images[j]:>4} min val')

    splits_created = 0

    for i in range(n_splits):
        print(f'\n{"="*70}')
        print(f'Split {i}')
        print(f'{"="*70}')

        # Phase 1: Greedy bin-packing
        cluster_stats = [(c, cluster_size_map[c]) for c in clusters]
        rng = np.random.default_rng(random_seed + i)
        rng.shuffle(cluster_stats)

        target_train = int(total_images * (1 - val_ratio))
        train_clusters = set()
        val_clusters = set()
        train_count = 0

        for cluster_id, size in cluster_stats:
            if train_count + size <= target_train:
                train_clusters.add(cluster_id)
                train_count += size
            else:
                gap_if_add = abs((train_count + size) - target_train)
                gap_if_skip = abs(train_count - target_train)
                if gap_if_add < gap_if_skip:
                    train_clusters.add(cluster_id)
                    train_count += size
                else:
                    val_clusters.add(cluster_id)

        val_count = sum(cluster_size_map[c] for c in val_clusters)
        print(f'  After bin-packing: train={len(train_clusters)} clusters ({train_count} imgs), '
              f'val={len(val_clusters)} clusters ({val_count} imgs)')

        # Phase 2: Enhanced repair — proportional thresholds + deficit-density scoring
        def get_val_class_counts(val_cls_set):
            counts = [0] * 12
            for c in val_cls_set:
                for j in range(12):
                    counts[j] += cluster_class_counts[c][j]
            return counts

        def get_underrepresented(val_counts):
            return {j for j in range(12) if val_counts[j] < min_val_images[j]}

        val_class_counts = get_val_class_counts(val_clusters)
        underrepresented = get_underrepresented(val_class_counts)

        if underrepresented:
            deficits = {j: min_val_images[j] - val_class_counts[j] for j in underrepresented}
            print(f'  Underrepresented in val (deficit): '
                  f'{", ".join(f"{CLASS_NAMES[j]}({deficits[j]})" for j in sorted(underrepresented))}')
            print(f'  Repairing...')

        max_val_count = int(total_images * 0.30)

        while underrepresented:
            deficits = {j: min_val_images[j] - val_class_counts[j] for j in underrepresented}

            best_cluster = None
            best_score = -1.0

            for c in list(train_clusters):
                c_size = cluster_size_map[c]

                # Safety cap: don't push val above 30%
                if val_count + c_size > max_val_count:
                    continue

                # Deficit-weighted density score
                deficit_score = 0.0
                for j in underrepresented:
                    c_class_count = cluster_class_counts[c][j]
                    if c_class_count > 0:
                        deficit_score += min(c_class_count, deficits[j]) / deficits[j]

                if deficit_score > 0:
                    score = deficit_score / c_size
                    if score > best_score:
                        best_score = score
                        best_cluster = c

            if best_cluster is None:
                print(f'  WARNING: Cannot repair further — no valid train cluster available '
                      f'(remaining: {", ".join(CLASS_NAMES[j] for j in sorted(underrepresented))})')
                break

            # Move best_cluster from train to val
            train_clusters.remove(best_cluster)
            val_clusters.add(best_cluster)
            c_size = cluster_size_map[best_cluster]
            val_count += c_size

            # Update val class counts
            for j in range(12):
                val_class_counts[j] += cluster_class_counts[best_cluster][j]

            covered = [f'{CLASS_NAMES[j]}(+{cluster_class_counts[best_cluster][j]})' 
                       for j in sorted(underrepresented) if cluster_class_counts[best_cluster][j] > 0]
            print(f'    Moved cluster {best_cluster} ({c_size} imgs, score={best_score:.4f}) '
                  f'to val — covers: {", ".join(covered)}')

            underrepresented = get_underrepresented(val_class_counts)

        train_df = merged[merged['cluster'].isin(train_clusters)]
        val_df = merged[merged['cluster'].isin(val_clusters)]

        print(f'  Final: train={len(train_clusters)} clusters ({len(train_df)} imgs, {len(train_df)/total_images:.1%}), '
              f'val={len(val_clusters)} clusters ({len(val_df)} imgs, {len(val_df)/total_images:.1%})')

        # Phase 3: Per-class coverage
        print(f'\n  {"Class":<15} {"Train Imgs":>12} {"Val Imgs":>12} {"Train Px%":>12} {"Val Px%":>12} {"Status":>10}')
        print(f'  {"-"*70}')

        missing = []
        warnings = []
        for j in range(12):
            col = CLASS_COLS[j]
            train_img_count = (train_df[col] > 0).sum()
            val_img_count = (val_df[col] > 0).sum()

            total_px = merged[col].sum()
            if total_px > 0:
                train_px_pct = train_df[col].sum() / total_px * 100
                val_px_pct = val_df[col].sum() / total_px * 100
            else:
                train_px_pct = 0
                val_px_pct = 0

            if train_img_count == 0:
                status = 'MISSING-T'
                missing.append((CLASS_NAMES[j], 'train'))
            elif val_img_count == 0:
                status = 'MISSING-V'
                missing.append((CLASS_NAMES[j], 'val'))
            elif val_img_count < min_val_images[j]:
                status = 'LOW'
            else:
                status = 'OK'

            if val_px_pct < 5.0 and val_img_count > 0:
                warnings.append((CLASS_NAMES[j], val_px_pct))

            print(f'  {CLASS_NAMES[j]:<15} {train_img_count:>12} {val_img_count:>12} {train_px_pct:>11.1f}% {val_px_pct:>11.1f}% {status:>10}')

        if warnings:
            print(f'\n  Low val coverage warnings:')
            for cls_name, pct in warnings:
                print(f'    - {cls_name}: only {pct:.1f}% of pixels in val')

        if missing:
            print(f'\n  WARNING: Split {i} skipped — missing classes:')
            for cls_name, split in missing:
                print(f'    - {cls_name} missing from {split}')
            continue

        # Save split files
        train_df[['ImageID']].to_csv(f'data/train_split_ids_{i}.csv', index=False)
        val_df[['ImageID']].to_csv(f'data/val_split_ids_{i}.csv', index=False)
        splits_created += 1
        print(f'\n  Saved data/train_split_ids_{i}.csv ({len(train_df)} rows)')
        print(f'  Saved data/val_split_ids_{i}.csv ({len(val_df)} rows)')

    print(f'\n{"="*70}')
    print(f'Done: {splits_created}/{n_splits} splits created successfully.')

In [132]:
# config = local_config
# config.init()
# analyze_clusters()

In [133]:
config = local_config
config.init()
generate_train_val_splits(n_splits=5, random_seed=3333)

Total images: 2621, Total clusters: 46
Per-class val thresholds (min_class_ratio=0.05):
  background        2621 total ->  131 min val
  person            2223 total ->  111 min val
  bike              1091 total ->   55 min val
  car                762 total ->   38 min val
  drone              264 total ->   13 min val
  boat                49 total ->    3 min val
  animal              73 total ->    4 min val
  obstacle          1732 total ->   87 min val
  construction      1455 total ->   73 min val
  vegetation        2414 total ->  121 min val
  road              2294 total ->  115 min val
  sky                388 total ->   19 min val

Split 0
  After bin-packing: train=44 clusters (2090 imgs), val=2 clusters (531 imgs)
  Underrepresented in val (deficit): drone(13), boat(3), animal(4), sky(19)
  Repairing...
    Moved cluster water_boat_with_wake (3 imgs, score=0.3684) to val — covers: boat(+3), sky(+2)
    Moved cluster animals_camels (2 imgs, score=0.3088) to val — covers: 

In [134]:
# config = local_config
# config.init()
# generate_cluster_labels_from_subfolders()

In [135]:
# config = local_config
# config.init()
# group_images_into_cluster_folders()

In [136]:
# config = local_config
# config.init()
# generate_image_clusters()